In [62]:
import pandas as pd
import numpy as np

In [63]:
# Csv
# Principal
ventas = pd.read_csv('ventas.csv')
# Apoyo
clientes = pd.read_csv('clientes.csv')
metodos_de_pago = pd.read_csv('metodos_pago.csv')
productos = pd.read_csv('productos.csv')

In [64]:
# Dimensiones del dataset
ventas.shape

(3029, 7)

In [65]:
# Dimensiones de los dataset de apoyo
print(clientes.shape)
print(metodos_de_pago.shape)
print(productos.shape)

(326, 6)
(5, 3)
(38, 5)


In [66]:
# VISTA PRELIMINAR DE LOS DATOS
ventas.head()

,ID_Venta,Fecha,ID_Cliente,ID_Producto,Cantidad,Método_Pago,Estado
0,919,31/01/2024,10,25,5,1,Completa
1,947,31/01/2024,106,5,1,4,Completa
2,1317,31/1/2024,235,25,3,3,Completa
3,1607,31/1/2024,114,15,5,1,Completa
4,2038,31/1/2024,132,2,5,4,Completa


In [67]:
ventas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3029 entries, 0 to 3028
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID_Venta     3029 non-null   int64 
 1   Fecha        3029 non-null   object
 2   ID_Cliente   3029 non-null   int64 
 3   ID_Producto  3029 non-null   int64 
 4   Cantidad     3029 non-null   int64 
 5   Método_Pago  3029 non-null   int64 
 6   Estado       3029 non-null   object
dtypes: int64(5), object(2)
memory usage: 165.8+ KB


In [68]:
ventas.describe()

,ID_Venta,ID_Cliente,ID_Producto,Cantidad,Método_Pago
count,3029.000000,3029.000000,3029.000000,3029.000000,3029.000000
mean,1492.663585,162.208320,19.675801,3.475404,3.359194
std,865.690540,94.276683,10.989542,1.702960,1.425749
min,1.000000,1.000000,1.000000,1.000000,1.000000
25%,729.000000,79.000000,10.000000,2.000000,2.000000
50%,1486.000000,162.000000,20.000000,3.000000,4.000000
75%,2243.000000,243.000000,29.000000,5.000000,5.000000
max,3000.000000,326.000000,38.000000,6.000000,5.000000


In [69]:
faltantes = ventas.isnull().sum()
faltantes

ID_Venta       0
Fecha          0
ID_Cliente     0
ID_Producto    0
Cantidad       0
Método_Pago    0
Estado         0
dtype: int64

In [70]:
# Valores faltantes en datasets de apoyo
faltantes_resumen = pd.DataFrame({
    'Dataset': ['Clientes', 'Métodos de Pago', 'Productos'],
    'Total_Registros': [len(clientes), len(metodos_de_pago), len(productos)],
    'Total_Valores_Faltantes': [
        clientes.isnull().sum().sum(),
        metodos_de_pago.isnull().sum().sum(), 
        productos.isnull().sum().sum()
    ]
})

faltantes_resumen

,Dataset,Total_Registros,Total_Valores_Faltantes
0,Clientes,326,0
1,Métodos de Pago,5,0
2,Productos,38,0


## Descripción General del Dataset
- Tamaño del Dataset: 3,029 registros de ventas con 7 variables
- Registros: Dataset completo sin valores faltantes
- Período: Datos de transacciones

### Variables Disponibles
- `ID_Venta` 
- `Fecha` 
- `ID_Cliente` 
- `ID_Producto` 
- `Cantidad` 
- `Método_Pago` 
- `Estado`

### Calidad de Datos
- No se detectaron valores nulos en ninguna columna
- No se identificaron problemas inmediatos de integridad de datos

### Tipos de Datos
- 5 variables numéricas (int64)
- 2 variables categóricas (object)

In [71]:
# Unicidad de llaves
print(f"ID_Venta: {ventas['ID_Venta'].nunique()} valores únicos de {len(ventas)} registros")

ID_Venta: 3000 valores únicos de 3029 registros


In [72]:
print(f"ID_Cliente: {ventas['ID_Cliente'].nunique()} clientes únicos")

ID_Cliente: 326 clientes únicos


In [73]:
# Ubicacion de los clientes
print(f"Regiones: {len(clientes['Región'].unique())}")
print(clientes['Región'].value_counts().sort_index())

Regiones: 6
Región
Buenos Aires    111
Centro           63
Cuyo             44
NEA              37
NOA               7
Patagonia        64
Name: count, dtype: int64


In [74]:
print(f"ID_Producto: {ventas['ID_Producto'].nunique()} productos únicos")

ID_Producto: 38 productos únicos


In [75]:
# Inventario de productos
productos.set_index("ID_Producto")

,Nombre_producto,Categoría,Precio_Unitario,Stock
ID_Producto,,,,
1,Leche,Lácteos,"12,24",3327
2,Yogur,Lácteos,"5,21",3358
3,Queso cremoso,Lácteos,"17,23",3167
4,Queso rallado,Lácteos,"19,23",2099
5,Manteca,Lácteos,"5,65",4929
6,Asado,Carnicería,"28,56",5137
7,Chorizo,Carnicería,"11,25",4068
8,Milanesa,Carnicería,"16,21",3140
9,Pollo,Carnicería,"18,56",4051


In [76]:
# Cantidad de productos por cada categoria
print(productos['Categoría'].value_counts().sort_index())

Categoría
Bebidas                4
Carnicería             6
Congelados             4
Conservas              4
Frutas y Verduras      6
Galletitas y Snacks    4
Lácteos                5
Panadería              5
Name: count, dtype: int64


Analisando el Excel se vio un problema:


Algunas fechas tienen formato: '31/01/2024' (DD/MM/YYYY), mientras que otras tienen: '31/1/2024' (D/M/YYYY), esto causa inconsistencia en la conversión.

In [77]:
# Convertir correctamente las fechas
ventas['Fecha'] = pd.to_datetime(ventas['Fecha'], dayfirst=True, errors='coerce')

# Mostrar el rango real corregido
fecha_min = ventas['Fecha'].min()
fecha_max = ventas['Fecha'].max()
print(f"Rango de fechas corregido: {fecha_min.strftime('%d/%m/%Y')} to {fecha_max.strftime('%d/%m/%Y')}")
print(f"Duracion real: {(fecha_max - fecha_min).days} dias")

Rango de fechas corregido: 31/01/2024 to 30/12/2024
Duracion real: 334 dias


In [78]:
# Métodos de pago
print(f"Métodos: {len(ventas['Método_Pago'].unique())}")
print(ventas['Método_Pago'].value_counts().sort_index())

Métodos: 5
Método_Pago
1    557
2    262
3    542
4    872
5    796
Name: count, dtype: int64


Significado de las 5 categorias de arriba

In [79]:
metodos_de_pago[['ID_Metodo', 'Método']].set_index("ID_Metodo")

,Método
ID_Metodo,
1,Efectivo
2,Tarjeta de Crédito
3,Tarjeta de Débito
4,Mercado Pago
5,Transferencia


In [80]:
# Análisis de estados
print("Estado de las transacciones: ")
print(ventas['Estado'].value_counts())

Estado de las transacciones: 
Estado
Completa     2548
Pendiente     471
Cancelada      10
Name: count, dtype: int64


In [81]:
# 5. Análisis de granularidad
transacciones_por_cliente = ventas.groupby('ID_Cliente').size()
print(f"Transacciones por cliente: {transacciones_por_cliente.min()}-{transacciones_por_cliente.max()}")
print(f"Promedio: {transacciones_por_cliente.mean():.1f} transacciones por cliente")

Transacciones por cliente: 2-19
Promedio: 9.3 transacciones por cliente


## Resumen Ejecutivo - Análisis de Datos

### Volúmenes y Unicidad
- ID_Venta: 3,000 valores únicos de 3,029 registros
- ID_Cliente: 326 clientes únicos
- ID_Producto: 38 productos únicos

### Distribución Geográfica
- Regiones: 6 regiones atendidas
  - Buenos Aires: 111 clientes
  - Patagonia: 64 clientes  
  - Centro: 63 clientes
  - Cuyo: 44 clientes
  - NEA: 37 clientes
  - NOA: 7 clientes

### Catálogo de Productos
- Categorías: 8 categorías de productos
  - Carnicería: 6 productos
  - Frutas y Verduras: 6 productos
  - Lácteos: 5 productos
  - Panadería: 5 productos
  - Bebidas: 4 productos
  - Congelados: 4 productos
  - Conservas: 4 productos
  - Galletitas y Snacks: 4 productos

### Período de Análisis
- Rango de fechas: 01/02/2024 to 09/09/2024
- Duración: 7 meses de datos

### Métodos de Pago
- Total métodos: 5 formas de pago
  - Mercado Pago: 872 transacciones
  - Transferencia: 796 transacciones
  - Efectivo: 557 transacciones
  - Tarjeta de Débito: 542 transacciones
  - Tarjeta de Crédito: 262 transacciones

### Estado de Transacciones
- Completa: 2,548 transacciones
- Pendiente: 471 transacciones 
- Cancelada: 10 transacciones 

### Comportamiento de Clientes
- Transacciones por cliente: 2-19 transacciones
- Promedio: 9.3 transacciones por cliente

In [82]:
# DETECCIÓN DE DUPLICADOS 
# Duplicados en ID_Venta
duplicados_id = ventas.duplicated(subset=['ID_Venta']).sum()
print(f"Duplicados en ID_Venta: {duplicados_id}")

# Duplicados semánticos (misma transacción)
duplicados_semanticos = ventas.duplicated(subset=['Fecha', 'ID_Cliente', 'ID_Producto', 'Cantidad']).sum()
print(f"Duplicados semánticos: {duplicados_semanticos}")

Duplicados en ID_Venta: 29
Duplicados semánticos: 29


In [83]:
print("RANGOS DE LOS DATOS:")

for columna in ventas.columns:
    if ventas[columna].dtype in ['int64', 'float64']:
        min_val = ventas[columna].min()
        max_val = ventas[columna].max()
        print(f"• {columna}:")
        print(f"  Mínimo: {min_val}")
        print(f"  Máximo: {max_val}")
        print(f"  Rango: {min_val} a {max_val}")

RANGOS DE LOS DATOS:
• ID_Venta:
  Mínimo: 1
  Máximo: 3000
  Rango: 1 a 3000
• ID_Cliente:
  Mínimo: 1
  Máximo: 326
  Rango: 1 a 326
• ID_Producto:
  Mínimo: 1
  Máximo: 38
  Rango: 1 a 38
• Cantidad:
  Mínimo: 1
  Máximo: 6
  Rango: 1 a 6
• Método_Pago:
  Mínimo: 1
  Máximo: 5
  Rango: 1 a 5


In [84]:
# CONSTRUCCIÓN DE VARIABLES DERIVADAS
ventas_transformadas = ventas.copy()

# Variables temporales
ventas_transformadas['Semana'] = ventas_transformadas['Fecha'].dt.isocalendar().week
ventas_transformadas['Mes'] = ventas_transformadas['Fecha'].dt.month
ventas_transformadas['Dia_Semana'] = ventas_transformadas['Fecha'].dt.day_name()
ventas_transformadas['Trimestre'] = ventas_transformadas['Fecha'].dt.quarter

print(" Variables creadas: Semana, Mes, Dia_Semana, Trimestre")

 Variables creadas: Semana, Mes, Dia_Semana, Trimestre


In [86]:
# UNIÓN CON DATASET PRODUCTOS Y CÁLCULO DE VENTA TOTAL
# Unir todos los datos de productos en una sola operación
ventas_completo = ventas_transformadas.merge(
    productos[['ID_Producto', 'Nombre_producto', 'Categoría', 'Precio_Unitario']], 
    on='ID_Producto', 
    how='left'
)

In [87]:
# CONVERTIR PRECIOS A FLOAT (agregar esta línea al inicio)
ventas_completo['Precio_Unitario'] = ventas_completo['Precio_Unitario'].str.replace(',', '.').astype(float)

# Calcular métricas por producto
ventas_completo['Venta_Total'] = ventas_completo['Cantidad'] * ventas_completo['Precio_Unitario']

resumen_productos = ventas_completo.groupby(['ID_Producto', 'Nombre_producto', 'Categoría']).agg(
    Unidades_Vendidas=('Cantidad', 'sum'),
    Venta_Total=('Venta_Total', 'sum'),
    Transacciones=('ID_Venta', 'count')
).reset_index()

# Calcular totales
total_unidades = resumen_productos['Unidades_Vendidas'].sum()
total_ventas = resumen_productos['Venta_Total'].sum()

# Calcular porcentajes
resumen_productos['%_Unidades'] = (resumen_productos['Unidades_Vendidas'] / total_unidades) * 100
resumen_productos['%_Venta'] = (resumen_productos['Venta_Total'] / total_ventas) * 100

# Ordenar por venta total descendente
resumen_productos = resumen_productos.sort_values('Venta_Total', ascending=False)

print(f"Total unidades vendidas: {total_unidades:,}")
print(f"Total ventas: ${total_ventas:,.2f}")

# Mostrar resultados
print("TOP 10 PRODUCTOS POR VENTAS:")
print(f"{'Producto':<25} {'Categoría':<20} {'Unidades':<10} {'Venta Total':<12} {'% Und':<8} {'% Venta':<8} {'Transacc':<10}")

for _, row in resumen_productos.head(10).iterrows():
    print(f"{row['Nombre_producto'][:24]:<25} {row['Categoría'][:19]:<20} "
          f"{row['Unidades_Vendidas']:<10,} ${row['Venta_Total']:<11,.2f} "
          f"{row['%_Unidades']:<7.1f}% {row['%_Venta']:<7.1f}% {row['Transacciones']:<10,}")

Total unidades vendidas: 10,527
Total ventas: $103,947.36
TOP 10 PRODUCTOS POR VENTAS:
Producto                  Categoría            Unidades   Venta Total  % Und    % Venta  Transacc  
Asado                     Carnicería           299        $8,539.44    2.8    % 8.2    % 81        
Milanesa                  Carnicería           320        $5,187.20    3.0    % 5.0    % 89        
Pizza congelada           Congelados           332        $5,129.40    3.2    % 4.9    % 86        
Queso rallado             Lácteos              259        $4,980.57    2.5    % 4.8    % 77        
Queso cremoso             Lácteos              273        $4,703.79    2.6    % 4.5    % 85        
Pollo                     Carnicería           252        $4,677.12    2.4    % 4.5    % 73        
Cerveza                   Bebidas              353        $4,073.62    3.4    % 3.9    % 99        
Empanadas                 Congelados           277        $3,750.58    2.6    % 3.6    % 75        
Hamburgesas c

In [88]:
# Resumen por categoría
print("RESUMEN POR CATEGORÍA:")
resumen_categoria = resumen_productos.groupby('Categoría').agg(
    Unidades_Vendidas=('Unidades_Vendidas', 'sum'),
    Venta_Total=('Venta_Total', 'sum'),
    Productos=('ID_Producto', 'count')
).reset_index()

resumen_categoria['%_Unidades'] = (resumen_categoria['Unidades_Vendidas'] / total_unidades) * 100
resumen_categoria['%_Venta'] = (resumen_categoria['Venta_Total'] / total_ventas) * 100
resumen_categoria = resumen_categoria.sort_values('Venta_Total', ascending=False)

# Tabla con nombres de columnas
print(f"{'Categoría':<20} {'Unidades':<10} {'Venta Total':<12} {'% Und':<8} {'% Venta':<8} {'# Productos':<12}")
for _, row in resumen_categoria.iterrows():
    print(f"{row['Categoría']:<20} {row['Unidades_Vendidas']:<10,} ${row['Venta_Total']:<11,.2f} "
          f"{row['%_Unidades']:<7.1f}% {row['%_Venta']:<7.1f}% {row['Productos']:<12}")

RESUMEN POR CATEGORÍA:
Categoría            Unidades   Venta Total  % Und    % Venta  # Productos 
Carnicería           1,673      $28,186.18   15.9   % 27.1   % 6           
Lácteos              1,381      $16,179.93   13.1   % 15.6   % 5           
Congelados           1,296      $15,034.41   12.3   % 14.5   % 4           
Panadería            1,270      $12,800.70   12.1   % 12.3   % 5           
Bebidas              1,182      $11,741.62   11.2   % 11.3   % 4           
Frutas y Verduras    1,479      $7,905.38    14.0   % 7.6    % 6           
Galletitas y Snacks  1,146      $7,165.18    10.9   % 6.9    % 4           
Conservas            1,100      $4,933.96    10.4   % 4.7    % 4           


In [89]:
# ANÁLISIS RFM (Recency, Frequency, Monetary)
fecha_referencia = ventas_completo['Fecha'].max() + pd.Timedelta(days=1)

# Filtrar solo transacciones completas para RFM
ventas_completas = ventas_completo[ventas_completo['Estado'] == 'Completa']

# Calcular métricas RFM por cliente
rfm_data = ventas_completas.groupby('ID_Cliente').agg({
    'Fecha': lambda x: (fecha_referencia - x.max()).days,  # Recency
    'ID_Venta': 'count',                                   # Frequency  
    'Venta_Total': 'sum'                                   # Monetary
}).reset_index()

rfm_data.columns = ['ID_Cliente', 'Recency', 'Frequency', 'Monetary']
# Mostrar estadísticas RFM
print("Estadísticas RFM:")
print(f"Recency (días desde última compra): {rfm_data['Recency'].mean():.1f} días en promedio")
print(f"Frequency (compras por cliente): {rfm_data['Frequency'].mean():.1f} compras en promedio")
print(f"Monetary (gasto por cliente): ${rfm_data['Monetary'].mean():.2f} en promedio")

Estadísticas RFM:
Recency (días desde última compra): 42.5 días en promedio
Frequency (compras por cliente): 7.8 compras en promedio
Monetary (gasto por cliente): $271.41 en promedio


In [91]:
# SEGMENTACIÓN RFM
# Crear segmentos RFM (usando cuartiles)
rfm_data['R_Score'] = pd.qcut(rfm_data['Recency'], 4, labels=[4, 3, 2, 1])  # Menor Recency = mejor
rfm_data['F_Score'] = pd.qcut(rfm_data['Frequency'], 4, labels=[1, 2, 3, 4])  # Mayor Frequency = mejor
rfm_data['M_Score'] = pd.qcut(rfm_data['Monetary'], 4, labels=[1, 2, 3, 4])   # Mayor Monetary = mejor

# Combinar scores
rfm_data['RFM_Score'] = rfm_data['R_Score'].astype(str) + rfm_data['F_Score'].astype(str) + rfm_data['M_Score'].astype(str)

# Crear segmentos basados en RFM
def asignar_segmento(rfm_score):
    r, f, m = int(rfm_score[0]), int(rfm_score[1]), int(rfm_score[2])
    if r == 4 and f == 4 and m == 4:
        return 'Campeones'
    elif r == 4 and f >= 3:
        return 'Clientes Leales'
    elif r >= 3:
        return 'Clientes con Potencial'
    elif r == 2:
        return 'Clientes en Riesgo'
    else:
        return 'Clientes Dormidos'

rfm_data['Segmento'] = rfm_data['RFM_Score'].apply(asignar_segmento)

# Mostrar distribución de segmentos
print("\nDistribución de segmentos de clientes:")
segmentos_dist = rfm_data['Segmento'].value_counts()
for segmento, count in segmentos_dist.items():
    porcentaje = (count / len(rfm_data)) * 100
    print(f"  {segmento}: {count} clientes ({porcentaje:.1f}%)")


Distribución de segmentos de clientes:
  Clientes con Potencial: 124 clientes (38.0%)
  Clientes en Riesgo: 82 clientes (25.2%)
  Clientes Dormidos: 81 clientes (24.8%)
  Clientes Leales: 20 clientes (6.1%)
  Campeones: 19 clientes (5.8%)


### Segmentación RFM de Clientes

Campeones (5.8%) - compran frecuentemente, recientemente y gastan mucho.


Clientes Leales (6.1%) - Compran regularmente y gastan bien, pero no tan recientemente. Estrategia: Programas de recompensas y ofertas exclusivas.


Clientes Potenciales (38.0%) - Compraron recientemente pero con menor frecuencia/gasto. Oportunidad de crecimiento.



Clientes en Riesgo (25.2%) - Antes eran buenos clientes pero no han comprado recientemente. 


Clientes Dormidos (24.8%) - No han comprado en mucho tiempo y tenían baja participación. 

In [94]:
# ANÁLISIS DE ESTACIONALIDAD Y PATRONES TEMPORALES

# Ventas por mes
ventas_mensuales = ventas_completo[ventas_completo['Estado'] == 'Completa'].groupby('Mes').agg({
    'Venta_Total': 'sum',
    'ID_Venta': 'count',
    'ID_Cliente': 'nunique'
}).reset_index()

ventas_mensuales.columns = ['Mes', 'Ventas_Totales', 'Transacciones', 'Clientes_Unicos']

print("VENTAS MENSUALES:")
print(f"{'Mes':<8} {'Ventas':<12} {'Transacc':<10} {'Clientes':<10} {'Ticket Prom':<12}")

for _, row in ventas_mensuales.iterrows():
    ticket_promedio = row['Ventas_Totales'] / row['Transacciones'] if row['Transacciones'] > 0 else 0
    print(f"{row['Mes']:<8} ${row['Ventas_Totales']:<11,.2f} {row['Transacciones']:<10} {row['Clientes_Unicos']:<10} ${ticket_promedio:<11.2f}")

VENTAS MENSUALES:
Mes      Ventas       Transacc   Clientes   Ticket Prom 
1.0      $255.98      7.0        7.0        $36.57      
2.0      $7,462.24    214.0      152.0      $34.87      
3.0      $8,863.88    236.0      173.0      $37.56      
4.0      $7,681.86    216.0      159.0      $35.56      
5.0      $7,589.69    228.0      164.0      $33.29      
6.0      $8,659.61    266.0      179.0      $32.55      
7.0      $7,716.68    232.0      167.0      $33.26      
8.0      $8,151.24    235.0      172.0      $34.69      
9.0      $8,210.48    227.0      169.0      $36.17      
10.0     $7,883.37    243.0      168.0      $32.44      
11.0     $7,035.33    214.0      150.0      $32.88      
12.0     $8,969.19    230.0      174.0      $39.00      


In [96]:
# Ventas por día de la semana
print("VENTAS POR DÍA DE LA SEMANA:")
ventas_diarias = ventas_completo[ventas_completo['Estado'] == 'Completa'].groupby('Dia_Semana').agg({
    'Venta_Total': 'sum',
    'ID_Venta': 'count',
    'ID_Cliente': 'nunique'
}).reset_index()

# Ordenar días de la semana lógico
dias_orden = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
ventas_diarias['Dia_Semana'] = pd.Categorical(ventas_diarias['Dia_Semana'], categories=dias_orden, ordered=True)
ventas_diarias = ventas_diarias.sort_values('Dia_Semana')

print(f"{'Día':<12} {'Ventas':<12} {'Transacc':<10} {'Clientes':<10} {'Ticket Prom':<12}")

for _, row in ventas_diarias.iterrows():
    ticket_promedio = row['Venta_Total'] / row['ID_Venta'] if row['ID_Venta'] > 0 else 0
    dia_es = {'Monday': 'Lunes', 'Tuesday': 'Martes', 'Wednesday': 'Miércoles', 
              'Thursday': 'Jueves', 'Friday': 'Viernes', 'Saturday': 'Sábado', 'Sunday': 'Domingo'}
    print(f"{dia_es[row['Dia_Semana']]:<12} ${row['Venta_Total']:<11,.2f} {row['ID_Venta']:<10} {row['ID_Cliente']:<10} ${ticket_promedio:<11.2f}")

VENTAS POR DÍA DE LA SEMANA:
Día          Ventas       Transacc   Clientes   Ticket Prom 
Lunes        $12,204.05   346        211        $35.27      
Martes       $12,906.52   345        210        $37.41      
Miércoles    $12,598.65   370        224        $34.05      
Jueves       $13,038.86   387        223        $33.69      
Viernes      $13,227.51   362        212        $36.54      
Sábado       $11,792.14   351        216        $33.60      
Domingo      $12,711.82   387        226        $32.85      
